## 1. Obtain the data

In this project there is a folder call data where I will use to put all the data, if you want to clone the repository you will not find it, but you can get the dataset from kaggle.

In [1]:
import pandas as pd


# Load the dataset
DATA_FOLDER = "../Marketing_Predict/data/"
FILE_PATH = DATA_FOLDER + "marketing_campaign_dataset.csv"

# Read the CSV file into a DataFrame
df = pd.read_csv(FILE_PATH)

# Display the first few rows of the DataFrame
df.head()


,Campaign_ID,Company,Campaign_Type,Target_Audience,Duration,Channel_Used,Conversion_Rate,Acquisition_Cost,ROI,Location,Language,Clicks,Impressions,Engagement_Score,Customer_Segment,Date
0,1,Innovate Industries,Email,Men 18-24,30 days,Google Ads,0.04,"$16,174.00",6.29,Chicago,Spanish,506,1922,6,Health & Wellness,2021-01-01
1,2,NexGen Systems,Email,Women 35-44,60 days,Google Ads,0.12,"$11,566.00",5.61,New York,German,116,7523,7,Fashionistas,2021-01-02
2,3,Alpha Innovations,Influencer,Men 25-34,30 days,YouTube,0.07,"$10,200.00",7.18,Los Angeles,French,584,7698,1,Outdoor Adventurers,2021-01-03
3,4,DataTech Solutions,Display,All Ages,60 days,YouTube,0.11,"$12,724.00",5.55,Miami,Mandarin,217,1820,7,Health & Wellness,2021-01-04
4,5,NexGen Systems,Email,Men 25-34,15 days,YouTube,0.05,"$16,452.00",6.50,Los Angeles,Mandarin,379,4201,3,Health & Wellness,2021-01-05


## 2. Initial exploration

In [2]:
# Check the columns in the dataset
print("Columns in the dataset:")
print(df.columns.tolist())

Columns in the dataset:
['Campaign_ID', 'Company', 'Campaign_Type', 'Target_Audience', 'Duration', 'Channel_Used', 'Conversion_Rate', 'Acquisition_Cost', 'ROI', 'Location', 'Language', 'Clicks', 'Impressions', 'Engagement_Score', 'Customer_Segment', 'Date']


In [3]:
# Check the type of data in each column
print("Data types:")
print(df.dtypes)

Data types:
Campaign_ID           int64
Company              object
Campaign_Type        object
Target_Audience      object
Duration             object
Channel_Used         object
Conversion_Rate     float64
Acquisition_Cost     object
ROI                 float64
Location             object
Language             object
Clicks                int64
Impressions           int64
Engagement_Score      int64
Customer_Segment     object
Date                 object
dtype: object


In [4]:
# Check the missing values in the dataset
print("Missing values in each column:")
print(df.isnull().sum())

Missing values in each column:
Campaign_ID         0
Company             0
Campaign_Type       0
Target_Audience     0
Duration            0
Channel_Used        0
Conversion_Rate     0
Acquisition_Cost    0
ROI                 0
Location            0
Language            0
Clicks              0
Impressions         0
Engagement_Score    0
Customer_Segment    0
Date                0
dtype: int64


## 3. Data cleaning

In [5]:
df_clean = df.copy()

# Normalize the column names by converting them to lowercase and replacing spaces with underscores

df_clean.columns= (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('[^a-z0-9_]', '', regex=True)
)

df_clean['acquisition_cost'] = df_clean['acquisition_cost'].str.replace('$', '').str.replace(',', '')

df_clean.head(5)

,campaign_id,company,campaign_type,target_audience,duration,channel_used,conversion_rate,acquisition_cost,roi,location,language,clicks,impressions,engagement_score,customer_segment,date
0,1,Innovate Industries,Email,Men 18-24,30 days,Google Ads,0.04,16174.00,6.29,Chicago,Spanish,506,1922,6,Health & Wellness,2021-01-01
1,2,NexGen Systems,Email,Women 35-44,60 days,Google Ads,0.12,11566.00,5.61,New York,German,116,7523,7,Fashionistas,2021-01-02
2,3,Alpha Innovations,Influencer,Men 25-34,30 days,YouTube,0.07,10200.00,7.18,Los Angeles,French,584,7698,1,Outdoor Adventurers,2021-01-03
3,4,DataTech Solutions,Display,All Ages,60 days,YouTube,0.11,12724.00,5.55,Miami,Mandarin,217,1820,7,Health & Wellness,2021-01-04
4,5,NexGen Systems,Email,Men 25-34,15 days,YouTube,0.05,16452.00,6.50,Los Angeles,Mandarin,379,4201,3,Health & Wellness,2021-01-05


In [6]:
# Change the data type of some columns.

df_clean['acquisition_cost'] = pd.to_numeric(df_clean['acquisition_cost'], errors='coerce').fillna(0)
df_clean['date'] = pd.to_datetime(df_clean['date'], errors='coerce')

df_clean.dtypes


campaign_id                  int64
company                     object
campaign_type               object
target_audience             object
duration                    object
channel_used                object
conversion_rate            float64
acquisition_cost           float64
roi                        float64
location                    object
language                    object
clicks                       int64
impressions                  int64
engagement_score             int64
customer_segment            object
date                datetime64[ns]
dtype: object

## 4. Create new columns



In [7]:
import numpy as np

# Calculate the click-through rate (CTR) and add it as a new column 'kpi_ctr_pct'
df_clean['kpi_ctr_pct'] = np.where(df_clean['impressions'] > 0, 
                                   (df_clean['clicks'] / df_clean['impressions']) * 100,
                                   0).round(4)

# Create time features from the 'date' column
df_clean['dim_year'] = df_clean['date'].dt.year
df_clean['dim_month'] = df_clean['date'].dt.month
df_clean['dim_month_name'] = df_clean['date'].dt.strftime('%B')
df_clean['dim_quarter'] = 'Q'+ df_clean['date'].dt.quarter.astype(str)
df_clean['dim_year_month'] = df_clean['date'].dt.to_period('M').astype(str)

df_clean.head(5)

,campaign_id,company,campaign_type,target_audience,duration,channel_used,conversion_rate,acquisition_cost,roi,location,...,impressions,engagement_score,customer_segment,date,kpi_ctr_pct,dim_year,dim_month,dim_month_name,dim_quarter,dim_year_month
0,1,Innovate Industries,Email,Men 18-24,30 days,Google Ads,0.04,16174.0,6.29,Chicago,...,1922,6,Health & Wellness,2021-01-01,26.3267,2021,1,January,Q1,2021-01
1,2,NexGen Systems,Email,Women 35-44,60 days,Google Ads,0.12,11566.0,5.61,New York,...,7523,7,Fashionistas,2021-01-02,1.5419,2021,1,January,Q1,2021-01
2,3,Alpha Innovations,Influencer,Men 25-34,30 days,YouTube,0.07,10200.0,7.18,Los Angeles,...,7698,1,Outdoor Adventurers,2021-01-03,7.5864,2021,1,January,Q1,2021-01
3,4,DataTech Solutions,Display,All Ages,60 days,YouTube,0.11,12724.0,5.55,Miami,...,1820,7,Health & Wellness,2021-01-04,11.9231,2021,1,January,Q1,2021-01
4,5,NexGen Systems,Email,Men 25-34,15 days,YouTube,0.05,16452.0,6.50,Los Angeles,...,4201,3,Health & Wellness,2021-01-05,9.0217,2021,1,January,Q1,2021-01


In [8]:
df_clean.describe()

,campaign_id,conversion_rate,acquisition_cost,roi,clicks,impressions,engagement_score,date,kpi_ctr_pct,dim_year,dim_month
count,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000.000000,200000,200000.000000,200000.0,200000.000000
mean,100000.500000,0.080070,12504.393040,5.002438,549.772030,5507.301520,5.494710,2021-07-01 23:35:09.600000,14.040550,2021.0,6.525480
min,1.000000,0.010000,5000.000000,2.000000,100.000000,1000.000000,1.000000,2021-01-01 00:00:00,1.005400,2021.0,1.000000
25%,50000.750000,0.050000,8739.750000,3.500000,325.000000,3266.000000,3.000000,2021-04-02 00:00:00,5.860550,2021.0,4.000000
50%,100000.500000,0.080000,12496.500000,5.010000,550.000000,5517.500000,5.000000,2021-07-02 00:00:00,9.978950,2021.0,7.000000
75%,150000.250000,0.120000,16264.000000,6.510000,775.000000,7753.000000,8.000000,2021-10-01 00:00:00,16.969825,2021.0,10.000000
max,200000.000000,0.150000,20000.000000,8.000000,1000.000000,10000.000000,10.000000,2021-12-31 00:00:00,99.202400,2021.0,12.000000
std,57735.171256,0.040602,4337.664545,1.734488,260.019056,2596.864286,2.872581,NaN,13.088122,0.0,3.447598


## 5. Create performance segments

In [9]:
# ROI Segmentation
df_clean['segment_roi'] = pd.cut(df_clean['roi'],
                                 bins=[-np.inf, 0, 3, 6, np.inf],
                                 labels=['Negative (<0)', 'Low (0-3)', 'Medium (3-6)', 'High (>6)']).astype(str)

# Acquisition Cost Segmentation
df_clean['segment_acquisition'] = pd.cut(df_clean['acquisition_cost'],
                                         bins=[-np.inf,5000, 10000, 15000, np.inf],
                                         labels=['Efficient (<5k)',
                                                 'Normal (5k-10k)',
                                                 'Expensive (10k-15k)',
                                                 'Very Expensive (>15k)']).astype(str)

df_clean.head(5)

,campaign_id,company,campaign_type,target_audience,duration,channel_used,conversion_rate,acquisition_cost,roi,location,...,customer_segment,date,kpi_ctr_pct,dim_year,dim_month,dim_month_name,dim_quarter,dim_year_month,segment_roi,segment_acquisition
0,1,Innovate Industries,Email,Men 18-24,30 days,Google Ads,0.04,16174.0,6.29,Chicago,...,Health & Wellness,2021-01-01,26.3267,2021,1,January,Q1,2021-01,High (>6),Very Expensive (>15k)
1,2,NexGen Systems,Email,Women 35-44,60 days,Google Ads,0.12,11566.0,5.61,New York,...,Fashionistas,2021-01-02,1.5419,2021,1,January,Q1,2021-01,Medium (3-6),Expensive (10k-15k)
2,3,Alpha Innovations,Influencer,Men 25-34,30 days,YouTube,0.07,10200.0,7.18,Los Angeles,...,Outdoor Adventurers,2021-01-03,7.5864,2021,1,January,Q1,2021-01,High (>6),Expensive (10k-15k)
3,4,DataTech Solutions,Display,All Ages,60 days,YouTube,0.11,12724.0,5.55,Miami,...,Health & Wellness,2021-01-04,11.9231,2021,1,January,Q1,2021-01,Medium (3-6),Expensive (10k-15k)
4,5,NexGen Systems,Email,Men 25-34,15 days,YouTube,0.05,16452.0,6.50,Los Angeles,...,Health & Wellness,2021-01-05,9.0217,2021,1,January,Q1,2021-01,High (>6),Very Expensive (>15k)


## 6. General summary

In [10]:
print("\n" + "="*60)
print("📋 GLOBAL KPI")
print("="*60)
print(f"   Average CTR:                {df_clean['kpi_ctr_pct'].mean():.2f}%")
print(f"   Average Conversion Rate:    {df_clean['conversion_rate'].mean():.4f}")
print(f"   Average Acquisition Cost:   ${df_clean['acquisition_cost'].mean():,.2f}")
print(f"   Average ROI:                {df_clean['roi'].mean():.2f}")
print(f"   Average Engagement Score:   {df_clean['engagement_score'].mean():.2f}/10")

print("\n📊 Average CTR per Channel:")
print(df_clean.groupby('channel_used')['kpi_ctr_pct'].mean().sort_values(ascending=False).round(4).to_string())

print("\n📊 Average ROI per Channel:")
print(df_clean.groupby('channel_used')['roi'].mean().sort_values(ascending=False).round(2).to_string())

print("\n📊 Average Acquisition Cost per Channel:")
print(df_clean.groupby('channel_used')['acquisition_cost'].mean().sort_values().round(2).to_string())


📋 GLOBAL KPI
   Average CTR:                14.04%
   Average Conversion Rate:    0.0801
   Average Acquisition Cost:   $12,504.39
   Average ROI:                5.00
   Average Engagement Score:   5.49/10

📊 Average CTR per Channel:
channel_used
YouTube       14.1196
Website       14.0971
Email         14.0543
Facebook      14.0499
Instagram     14.0037
Google Ads    13.9190

📊 Average ROI per Channel:
channel_used
Facebook      5.02
Website       5.01
Google Ads    5.00
Email         5.00
YouTube       4.99
Instagram     4.99

📊 Average Acquisition Cost per Channel:
channel_used
YouTube       12481.39
Website       12487.81
Instagram     12491.76
Facebook      12510.90
Email         12526.39
Google Ads    12528.03


## 7. Create CSV to create dashboard.

In [13]:
output_main = "..\Marketing_Predict\data\marketing_kpis_looker.csv"
df_clean.drop(columns=['date'], errors='ignore').to_csv(output_main, index=False)
print(f"'{output_main}' ({len(df_clean):,} rows)")

# Summary aggregated by channel and campaign type
df_summary = df_clean.groupby(['channel_used', 'campaign_type']).agg(
    avg_ctr = ('kpi_ctr_pct', 'mean'),
    avg_conversion_rate = ('conversion_rate', 'mean'),
    avg_acquisition_cost = ('acquisition_cost', 'mean'),
    avg_roi = ('roi', 'mean'),
    avg_engagement = ('engagement_score', 'mean'),
    total_clicks = ('clicks', 'sum'),
    total_impressions = ('impressions', 'sum'),
    campaign_count = ('campaign_id', 'count')
)
df_summary.to_csv("..\Marketing_Predict\data\marketing_summary_by_channel.csv")
print("'marketing_summary_by_channel.csv' → Summary by channel")

'..\Marketing_Predict\data\marketing_kpis_looker.csv' (200,000 rows)
'marketing_summary_by_channel.csv' → Summary by channel
